# 04 — Feature ablation

Does the feature engineering earn its complexity? One XGBoost classifier with
default hyperparameters is trained four times on progressively larger feature
sets drawn from KNHANES, and the four are compared on a common held-out split.

| Set | Features |
|---|---|
| baseline | 9 routinely measured variables |
| baseline + interactions | 57 |
| extended | 17 measured variables |
| extended + interactions | 241 |

Each model is also scored by its **Contributing Feature Ratio** — the share of
input features the model actually splits on. It asks whether a larger feature set
is being used or merely carried.

Constant columns are excluded from both sides of that ratio. A constant column
cannot contribute by construction, so counting it as a non-contributing feature
penalises the model for a column that holds no information. Here `RACE` is
constant within a single cohort, and in the 9- and 17-feature runs it is the only
feature with zero importance.

These runs read the `RACE = 1` feature table. The value is immaterial to the
result — `RACE` is constant within a single cohort and these runs use default
hyperparameters, so no tree can split on it and no column sampling reaches it.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import polars as pl

from src.data.io import output_path, processed_path
from src.features.selection import ablation_feature_sets, drop_target_derived
from src.logging_utils import configure_logging
from src.models.evaluate import compute_metrics, contributing_feature_ratio
from src.models.train import split_frames, train_classifier

configure_logging(ROOT / "logs")

features = drop_target_derived(
    pl.read_parquet(processed_path("KNHANES_race1_features.parquet"))
)
feature_sets = ablation_feature_sets(features)
print(f"after dropping identifiers and target-derived columns: {features.shape}")

2026-09-22 19:47:20 [INFO] src.features.selection: baseline: 9 features


2026-09-22 19:47:20 [INFO] src.features.selection: baseline + interactions: 57 features


2026-09-22 19:47:20 [INFO] src.features.selection: extended: 17 features


2026-09-22 19:47:20 [INFO] src.features.selection: extended + interactions: 241 features


after dropping identifiers and target-derived columns: (15138, 242)


## Train and evaluate

Each set is split 70/30 with stratification and a fixed seed, so all four models
are scored on the same participants. Class imbalance is handled by weighting —
`scale_pos_weight` — rather than by resampling.

In [2]:
runs = {}
rows = []
for name, frame in feature_sets.items():
    run = train_classifier(frame, algorithm="XGBoost")
    metrics = compute_metrics(run["y_test"], run["preds"])
    cfr = contributing_feature_ratio(
        run["model"], list(run["X_train"].columns), run["X_train"]
    )
    runs[name] = run
    rows.append(
        {
            "feature_set": name,
            "n_features": frame.width - 1,
            **{key: value for key, value in metrics.items() if key != "confusion_matrix"},
            "CFR": cfr["ratio"],
            "contributing_features": cfr["contributing"],
            "features_considered": cfr["total"],
            **metrics["confusion_matrix"],
        }
    )

# Ordered smallest to largest within each variable set.
ablation = pl.DataFrame(rows).drop("optimal_threshold")
ablation.to_pandas().to_excel(output_path("feature_ablation.xlsx"), index=False)
ablation

2026-09-22 19:47:20 [INFO] src.models.train: XGBoost on 9 features: 10596 training / 4542 test rows


2026-09-22 19:47:20 [INFO] src.models.evaluate: CFR: 8/8 contributing (1 constant column(s) excluded)


2026-09-22 19:47:20 [INFO] src.models.train: XGBoost on 57 features: 10596 training / 4542 test rows


2026-09-22 19:47:20 [INFO] src.models.evaluate: CFR: 38/56 contributing (1 constant column(s) excluded)


2026-09-22 19:47:20 [INFO] src.models.train: XGBoost on 17 features: 10596 training / 4542 test rows


2026-09-22 19:47:21 [INFO] src.models.evaluate: CFR: 16/16 contributing (1 constant column(s) excluded)


2026-09-22 19:47:21 [INFO] src.models.train: XGBoost on 241 features: 10596 training / 4542 test rows


2026-09-22 19:47:22 [INFO] src.models.evaluate: CFR: 198/240 contributing (1 constant column(s) excluded)


feature_set,n_features,roc_auc,accuracy,sensitivity(recall),specificity,PPV(precision),NPV,f1_score,Youdens_Index,pr_auc,CFR,contributing_features,features_considered,TP,TN,FP,FN
str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64
"""baseline""",9,0.847,0.785,0.699,0.818,0.598,0.875,0.645,0.517,0.709,1.0,8,8,886,2679,595,382
"""baseline + interactions""",57,0.842,0.791,0.662,0.84,0.616,0.865,0.639,0.503,0.701,0.679,38,56,840,2751,523,428
"""extended""",17,0.856,0.801,0.694,0.842,0.63,0.877,0.66,0.536,0.723,1.0,16,16,880,2757,517,388
"""extended + interactions""",241,0.857,0.807,0.64,0.871,0.658,0.862,0.649,0.511,0.735,0.825,198,240,811,2853,421,457


Adding the eight further measured variables buys more than adding interaction
terms does: the 17-variable set beats the 9-variable set on every metric, while
the interaction terms mostly trade sensitivity for specificity at roughly
constant AUC.